# Reset / Replay
# 0. 介绍

**研究背景**：Agent 在执行任务时会修改文件、缓存和其他环境状态。为了公平比较多次运行，或准确复现某次运行，外层程序需要先把环境恢复到同一个已知起点，再按相同记录重新执行。这个过程就是 Reset / Replay。

**现存问题**：如果新一轮任务直接使用上一轮留下的环境，旧文件、旧缓存或旧状态就会混入本次结果。这样，即使大模型给出了相同的正确操作，最终产物也可能不同，造成错误通过、错误失败，或者无法复现问题。

**解决方案**：本 Notebook 将实现一个极简的 Reset / Replay，在每次运行前恢复并校验固定的环境基线，再重放同一份真实 API 决策。然后对比两种做法：基线版本直接在污染环境中执行而产生多余残留，改进版本从干净环境重放并得到预期产物，从而直观看到环境重置与重放如何保证多次运行相互隔离、结果可比较且问题可复现。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 定义可用工具
大模型不能直接修改外部环境，只能告诉程序要执行什么动作。这里提供一个 `write_file` 工具，让大模型给出文件名和文件内容。

In [2]:
tools = [{
    "type": "function",
    "function": {
        "name": "write_file",
        "description": "在工作区中写入一个文件",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "文件名"},
                "content": {"type": "string", "description": "文件内容"},
            },
            "required": ["path", "content"],
        },
    },
}]
print("可用工具：", tools[0]["function"]["name"])

可用工具： write_file


输出显示 `write_file` 已经准备好，说明大模型可以要求程序写入文件，但此时工作区还没有发生变化。下一步会给大模型一个具体任务。

## 2.2 写出具体任务
为了让前后两种做法可以公平比较，这里固定一项简单任务：创建一个内容为 `42` 的 `answer.txt`。后续只获取一次真实模型决策，再把同一个决策用于两种做法。

In [3]:
messages = [
    {"role": "system", "content": "请调用工具完成任务，不要只回复文字。"},
    {"role": "user", "content": "创建 answer.txt，文件内容必须是 42。"},
]
print("任务：", messages[1]["content"])

任务： 创建 answer.txt，文件内容必须是 42。


输出显示了大模型将要完成的任务。下一步会固定任务开始前的环境基线和任务完成后的预期结果。

## 2.3 准备环境基线与目标
Reset 必须知道环境原本是什么样，才能清除上一轮留下的状态。这里用字典表示一个极简工作区：空字典是固定基线，只包含目标文件的字典是预期结果。

In [4]:
baseline_files = {}
expected_files = {"answer.txt": "42"}
workspace = baseline_files.copy()

print("环境基线：", baseline_files)
print("预期结果：", expected_files)

环境基线： {}
预期结果： {'answer.txt': '42'}


输出显示任务必须从空工作区开始，完成后只能存在 `answer.txt`。下一步会定义如何把一条已经记录的模型动作重新执行到工作区中。

## 2.4 定义动作重放方式
Replay 的核心是按记录再次执行相同动作。这里的动作会包含文件名和文件内容，重放函数只负责把它们写入当前工作区。

In [5]:
def replay_action(files, action):
    # 按模型给出的记录写入文件
    files[action["path"]] = action["content"]


print("动作重放方式已定义，工作区仍为：", workspace)

动作重放方式已定义，工作区仍为： {}


输出显示动作重放方式已经准备好，但还没有收到真实模型动作，所以工作区仍然为空。下一步会定义任务成功的统一标准。

## 2.5 定义成功标准
只检查目标文件存在还不够，因为上一轮遗留的文件仍可能混入结果。这里要求整个工作区与预期结果完全相同，后续两种做法都使用这一个标准。

In [6]:
def grade(files):
    return files == expected_files


print("成功标准：工作区必须等于", expected_files)

成功标准：工作区必须等于 {'answer.txt': '42'}


输出显示了唯一的成功标准，说明后面的基线版本和改进版本会用同一把尺子判断结果。至此，工具、任务、环境基线、动作重放方式和成功标准都已准备完成，下一章将调用真实大模型并保存它给出的动作。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
模型、工具和任务已经准备好，现在把它们一起发给真实大模型。这里要求模型必须选择工具，并记录从发出请求到收到回复所用的时间。

In [7]:
from time import perf_counter

start = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",  # 必须选择一个工具
    temperature=0,
)
latency_ms = round((perf_counter() - start) * 1000)
raw_real = response.model_dump()

print(f"模型来源：{config['NANO_BACKEND']}")
print(f"模型名称：{model_name}")
print(f"等待时间：{latency_ms} ms")

模型来源：openai
模型名称：LongCat-2.0
等待时间：3601 ms


输出显示了模型来源、模型名称和等待时间，说明真实大模型已经返回结果，完整响应保存在 `raw_real` 中。下一步会从响应中取出可重复执行的具体动作。

## 3.2 查看并保存动作
外部环境需要明确的文件名和文件内容才能执行操作。这里读取模型的第一条工具调用，把参数保存为 `action`，同时显示停止原因和 Token 用量。

In [8]:
import json

choice = raw_real["choices"][0]
tool_call = choice["message"]["tool_calls"][0]
arguments = json.loads(tool_call["function"]["arguments"])
action = {"path": arguments["path"], "content": arguments["content"]}
usage = raw_real["usage"]

print(f"工具：{tool_call['function']['name']}")
print(f"动作：{action}")
print(f"停止原因：{choice['finish_reason']}")
print(f"Token 用量：输入 {usage['prompt_tokens']}，输出 {usage['completion_tokens']}，总计 {usage['total_tokens']}")

工具：write_file
动作：{'path': 'answer.txt', 'content': '42'}
停止原因：tool_calls
Token 用量：输入 184，输出 71，总计 255


输出显示大模型选择了 `write_file`，并给出 `answer.txt` 的文件内容，说明模型已经生成了正确动作。`action` 保存了后续两种做法共用的真实决策；停止原因、Token 用量和等待时间记录了本次请求的运行情况，下一章将定义不重置环境的基线组件。

# 4. 定义基线组件
最简单的重放方式，是把保存的动作直接执行到当前工作区。这里定义一个基线函数：它会重放动作，但不会先把工作区恢复到固定基线。

In [9]:
def replay_only(files, action):
    # 直接使用当前环境，不恢复基线
    replay_action(files, action)
    return files


print("基线组件已定义：直接重放，不恢复环境")

基线组件已定义：直接重放，不恢复环境


输出说明基线组件已经定义完成，但还没有执行动作或修改工作区。下一章会先放入上一轮遗留文件，再使用这个组件重放第 3 章保存的真实动作。

# 5. 展示基线故障
## 5.1 模拟上一轮遗留状态
连续运行任务时，当前工作区可能还保留上一轮产生的文件。这里加入一个 `debug.log`，表示新一轮任务开始前没有被清除的旧状态。

In [10]:
workspace["debug.log"] = "上一轮留下的文件"
print("当前工作区：", workspace)

当前工作区： {'debug.log': '上一轮留下的文件'}


输出显示任务尚未开始，工作区就已经存在 `debug.log`。下一步会让基线组件直接在这个环境中重放真实模型动作。

## 5.2 直接重放真实动作
基线组件不会恢复环境，因此会保留 `debug.log`，再执行模型给出的写文件动作。这里同时保存重放前后的工作区，方便直接比较状态变化。

In [11]:
baseline_before = workspace.copy()
replay_only(workspace, action)
baseline_after = workspace.copy()

print("重放前：", baseline_before)
print("真实动作：", action)
print("重放后：", baseline_after)

重放前： {'debug.log': '上一轮留下的文件'}
真实动作： {'path': 'answer.txt', 'content': '42'}
重放后： {'debug.log': '上一轮留下的文件', 'answer.txt': '42'}


输出显示真实模型动作正确创建了 `answer.txt`，但旧的 `debug.log` 也被保留下来。下一步会用第 2 章定义的统一标准判断完整结果。

## 5.3 判断基线结果
成功标准要求工作区只能包含目标文件。这里比较实际工作区与预期结果，确认旧状态是否影响了本次任务。

In [12]:
baseline_passed = grade(baseline_after)

print("预期结果：", expected_files)
print("实际结果：", baseline_after)
print("任务通过：", baseline_passed)

预期结果： {'answer.txt': '42'}
实际结果： {'debug.log': '上一轮留下的文件', 'answer.txt': '42'}
任务通过： False


输出为 `False`，说明基线做法没有通过。大模型给出的文件名和内容都正确，失败来自上一轮遗留文件混入了本次结果；下一章将定义一个在重放前恢复环境基线的改进组件。

# 6. 定义改进组件
旧环境中可能留下很多未知状态，逐个删除容易遗漏。更可靠的做法是把固定基线作为唯一起点：每次运行都从基线创建一个新工作区，再重放保存的动作。

In [13]:
def reset_and_replay(baseline, action):
    clean_files = baseline.copy()  # Reset：从固定基线创建新环境
    replay_action(clean_files, action)  # Replay：重新执行同一动作
    return clean_files


print("改进组件已定义：先恢复基线，再重放动作")

改进组件已定义：先恢复基线，再重放动作


输出说明改进组件已经定义完成，但本节还没有运行它，原来的污染工作区也没有改变。下一章将使用这个组件重放与基线版本完全相同的真实模型动作。

# 7. 展示修复结果
## 7.1 恢复基线并重放动作
当前工作区仍包含上一轮遗留状态。这里不再复用它，而是让改进组件从固定基线创建新工作区，再重放与基线版本完全相同的真实模型动作。

In [14]:
fixed_before = workspace.copy()
fixed_after = reset_and_replay(baseline_files, action)

print("恢复前：", fixed_before)
print("环境基线：", baseline_files)
print("真实动作：", action)
print("重放后：", fixed_after)

恢复前： {'debug.log': '上一轮留下的文件', 'answer.txt': '42'}
环境基线： {}
真实动作： {'path': 'answer.txt', 'content': '42'}
重放后： {'answer.txt': '42'}


输出显示恢复前的工作区含有旧文件，固定基线则为空。改进组件从这个空基线重放同一动作后，只生成了 `answer.txt`，`debug.log` 没有进入新结果。下一步会使用统一标准判断任务是否通过。

## 7.2 判断修复结果
基线版本和改进版本必须使用同一把尺子。这里仍然要求整个工作区与预期结果完全相同。

In [15]:
fixed_passed = grade(fixed_after)

print("预期结果：", expected_files)
print("实际结果：", fixed_after)
print("任务通过：", fixed_passed)

预期结果： {'answer.txt': '42'}
实际结果： {'answer.txt': '42'}
任务通过： True


输出为 `True`，说明改进组件完成了任务。两种做法使用同一个真实模型动作和成功标准，结果差异只来自重放前是否恢复固定基线；下一章将汇总完整对照。

# 8. 汇总消融对照
## 8.1 对比两种做法
只改变重放前是否恢复环境基线，再并排比较结果，就能看出 Reset 的作用。这里先汇总两种做法共用的真实 API 信息，再展示各自的重放起点、最终状态和任务结果。

In [16]:
shared_info = {
    "模型来源": config["NANO_BACKEND"],
    "模型": model_name,
    "真实动作": action,
    "API 调用": 1,
    "等待时间（毫秒）": latency_ms,
    "Token 总量": usage["total_tokens"],
    "停止原因": choice["finish_reason"],
}
comparison = [
    {"做法": "直接重放", "重放起点": baseline_before, "最终状态": baseline_after, "任务通过": baseline_passed},
    {"做法": "Reset / Replay", "重放起点": baseline_files, "最终状态": fixed_after, "任务通过": fixed_passed},
]

print("共同信息：")
print(json.dumps(shared_info, ensure_ascii=False, indent=2))
print("消融对照：")
print(json.dumps(comparison, ensure_ascii=False, indent=2))

共同信息：
{
  "模型来源": "openai",
  "模型": "LongCat-2.0",
  "真实动作": {
    "path": "answer.txt",
    "content": "42"
  },
  "API 调用": 1,
  "等待时间（毫秒）": 3601,
  "Token 总量": 255,
  "停止原因": "tool_calls"
}
消融对照：
[
  {
    "做法": "直接重放",
    "重放起点": {
      "debug.log": "上一轮留下的文件"
    },
    "最终状态": {
      "debug.log": "上一轮留下的文件",
      "answer.txt": "42"
    },
    "任务通过": false
  },
  {
    "做法": "Reset / Replay",
    "重放起点": {},
    "最终状态": {
      "answer.txt": "42"
    },
    "任务通过": true
  }
]


输出显示两种做法共用同一次真实 API 调用和同一个模型动作。直接重放从污染状态开始，最终保留了 `debug.log`，任务失败；Reset / Replay 从固定空基线开始，最终只生成 `answer.txt`，任务通过。模型没有改变，决定结果的是模型外层是否为每次重放提供干净且一致的起点，至此本 Notebook 的对照实验结束。

## 8.2 拓展

### nano 版省略了什么

nano 版只重置一个内存字典并重放单个动作，没有覆盖文件系统快照、容器镜像、外部服务、随机数、时钟、网络响应、幂等键、事件日志压缩和跨版本迁移。生产 Replay 必须记录足以重建决策环境的输入与版本，同时区分可安全重放和不可逆副作用。

### 延伸阅读


1. 2025, [Anthropic, Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)：跨上下文窗口的初始化、增量进展与交接产物。
2. 2025, [Anthropic, Managing context on the Claude Developer Platform](https://claude.com/blog/context-management)：长运行任务中的状态保留、清理与外部记忆。
3. 2026, [Code as Agent Harness](https://arxiv.org/abs/2605.18747)：把可执行代码、状态和验证组合成可复现 Harness。